# Getting Started

tsam_xarray wraps [tsam](https://github.com/FZJ-IEK3-VSA/tsam) for xarray DataArrays.
This notebook shows the basic workflow.

## Sample data

tsam_xarray includes sample energy data with realistic profiles for documentation.

In [ ]:
import plotly.io as pio
import xarray_plotly  # noqa: F401 — registers .plotly accessor

import tsam_xarray
from tsam_xarray._sample_data import sample_energy_data

pio.renderers.default = "notebook"

da = sample_energy_data(n_days=30)
print(f"Shape: {dict(da.sizes)}")
da.sel(region="north", scenario="low").plotly.line(
    x="time", color="variable", title="Input data (north, low)"
)

In [ ]:
da.sel(region="north", scenario="low").to_dataframe("value").head()

## Aggregate

For a `(time, variable)` array, `cluster_dim` is auto-detected.

In [ ]:
da_simple = da.sel(region="north", scenario="low")

result = tsam_xarray.aggregate(
    da_simple,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
)
result.typical_periods.to_dataframe("value").head(10)

In [ ]:
result.typical_periods.plotly.line(
    x="timestep", facet_col="variable", color="cluster", title="Typical periods"
)

## Inspect results

The result contains xarray-native fields.

In [ ]:
print(f"Clusters: {result.n_clusters}")
print(f"Timesteps per period: {result.n_timesteps_per_period}")
print("Cluster weights (days each represents):")
result.cluster_weights.to_dataframe("weight")

In [ ]:
result.accuracy.rmse.to_dataframe("RMSE")

## Reconstructed vs original

In [ ]:
import xarray as xr

comparison = xr.concat(
    [da_simple.sel(variable="solar"), result.reconstructed.sel(variable="solar")],
    dim="source",
).assign_coords(source=["original", "reconstructed"])
comparison.plotly.line(
    x="time", color="source", title="Original vs reconstructed (solar)"
)

In [ ]:
result.residuals.plotly.line(x="time", facet_col="variable", title="Residuals")

## Passing tsam parameters

All `tsam.aggregate()` keyword arguments pass through.

In [ ]:
from tsam import ClusterConfig

result_km = tsam_xarray.aggregate(
    da_simple,
    time_dim="time",
    cluster_dim="variable",
    n_clusters=4,
    cluster=ClusterConfig(method="kmeans"),
)
result_km.accuracy.rmse.to_dataframe("RMSE")

## Disaggregate

`disaggregate()` is the inverse of `aggregate()` — it maps any data on the `(cluster, timestep)` grid back to the original time axis.

In [ ]:
disaggregated = result.disaggregate(result.typical_periods)

comparison = xr.concat(
    [da_simple.sel(variable="solar"), disaggregated.sel(variable="solar")],
    dim="source",
).assign_coords(source=["original", "disaggregated"])
comparison.plotly.line(
    x="time", color="source", title="Disaggregated vs original (solar)"
)